# 06c — Extended Downstream Evaluation Adapted from Friend's Notebook

Fixed v3: **single-run notebook**.

The first big code cell only defines functions and configuration. It does **not** start evaluation.

Only the final cell starts:

```python
outputs = run_project_evaluation(...)
```

This version keeps your friend's two-framework design:

```text
FRAMEWORKS = ["tsai", "aeon"]
MODALITIES = ["acc", "bvp", "eda", "temp", "fused"]
```

and adapts it to:

```text
KoVAE: rollout_v1, posterior_bank_v2
TimeVAE: prior_v1
```


In [1]:
from pathlib import Path
import inspect
import itertools
import json
import random
import re
import warnings
import shutil

import joblib
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_recall_fscore_support,
    precision_score,
    recall_score,
)

# =========================
# MAIN RUN KNOBS
# =========================

RUN_REAL_TO_REAL_ONLY = False
USE_REAL_MODELS = True
FRAMEWORKS = ["tsai", "aeon"]

# =========================
# BASIC SETTINGS
# =========================

MODALITIES = ["acc", "bvp", "eda", "temp", "fused"]

EPOCHS_TSAI = 50
BATCH_SIZE = 1024
LR_TSAI = 1e-3
TSAI_ARCH = "InceptionTimePlus"

AEON_N_KERNELS = 5000
AEON_N_JOBS = 8

SEED = 42
SYN_TEST_N_SUBJECTS = 3

OUT_DIR = Path("tsai_aeon_results")
REAL_DIR = Path("processed_all_subjects_native_rates")
SYN_DIR = Path("synth_data")

OUT_DIR.mkdir(parents=True, exist_ok=True)

# =========================
# PATHS AND CONSTANTS
# =========================

REAL_X_ACC_PATH = REAL_DIR / "all_X_acc_32hz.npy"
REAL_X_BVP_PATH = REAL_DIR / "all_X_bvp_64hz.npy"
REAL_X_SLOW_PATH = REAL_DIR / "all_X_slow_4hz.npy"
REAL_Y_PATH = REAL_DIR / "all_y.npy"
REAL_SUBJECT_PATH = REAL_DIR / "all_subject.npy"

SYN_X_ACC_PATH = SYN_DIR / "generated_subjects_X_acc_32hz.npy"
SYN_X_BVP_PATH = SYN_DIR / "generated_subjects_X_bvp_64hz.npy"
SYN_X_SLOW_PATH = SYN_DIR / "generated_subjects_X_slow_4hz.npy"
SYN_Y_PATH = SYN_DIR / "generated_subjects_all_y.npy"
SYN_SUBJECT_PATH = SYN_DIR / "generated_subjects_all_subject.npy"

ACTIVITY_IDS = [1, 2, 3, 4, 5, 6, 7, 8]
LABEL_TO_INDEX = {label: i for i, label in enumerate(ACTIVITY_IDS)}
INDEX_TO_LABEL = {i: label for label, i in LABEL_TO_INDEX.items()}

TRAIN_SUBJECTS = ["S1", "S2", "S3", "S4", "S5", "S6", "S9", "S11", "S12", "S13"]
VAL_SUBJECTS = ["S14", "S15"]
TEST_SUBJECTS = ["S7", "S8", "S10"]

MODALITY_INFO = {
    "acc": {
        "real_path": REAL_X_ACC_PATH,
        "syn_path": SYN_X_ACC_PATH,
        "expected_shape_tail": (256, 3),
        "native_hz": 32,
        "channel_names": ["ACC_x", "ACC_y", "ACC_z"],
    },
    "bvp": {
        "real_path": REAL_X_BVP_PATH,
        "syn_path": SYN_X_BVP_PATH,
        "expected_shape_tail": (512, 1),
        "native_hz": 64,
        "channel_names": ["BVP"],
    },
    "slow": {
        "real_path": REAL_X_SLOW_PATH,
        "syn_path": SYN_X_SLOW_PATH,
        "expected_shape_tail": (32, 2),
        "native_hz": 4,
        "channel_names": ["EDA", "TEMP"],
    },
}

FUSED_MODALITY_NAME = "fused"
FUSED_TARGET_LEN = 512
FUSED_NATIVE_HZ = 64
FUSED_CHANNEL_NAMES = ["ACC_x", "ACC_y", "ACC_z", "BVP", "EDA", "TEMP"]

# =========================
# PROJECT ADAPTATION SETTINGS
# =========================

PROJECT_ROOT = Path("/home/iailab42/khans1/projects/ir")
PRETRAINED_RESULTS_DIR = PROJECT_ROOT / "models/downstream/pretrained/tsai_aeon_results"

CURRENT_MODEL_FAMILY = ""
CURRENT_SYNTHETIC_METHOD = ""
CURRENT_SYNTHETIC_METHOD_DISPLAY = ""

MODEL_FAMILY_CONFIGS = {
    "kovae": {
        "synthetic_base_dir": PROJECT_ROOT / "data/synthetic_subjects/kovae",
        "synthetic_methods": ["rollout_v1", "posterior_bank_v2"],
        "method_display_names": {
            "rollout_v1": "KoVAE-Rollout",
            "posterior_bank_v2": "KoVAE-Posterior",
        },
    },
    "timevae": {
        "synthetic_base_dir": PROJECT_ROOT / "data/synthetic_subjects/timevae",
        "synthetic_methods": ["prior_v1"],
        "method_display_names": {
            "prior_v1": "TimeVAE-Prior",
        },
    },
}

# aeon compatibility fix used only for temporary classifier arrays.
AEON_FIX_LOW_VARIATION = True
AEON_MIN_STD = 1e-6


# =========================
# HELPERS
# =========================

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)


def require_file(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Missing file: {path}")
    return path


def subject_sort_key(s):
    s = str(s)
    if s.startswith("S") and s[1:].isdigit():
        return ("S", int(s[1:]))
    m = re.search(r"(\d+)$", s)
    if m is not None:
        return (s[:m.start(1)], int(m.group(1)))
    return (s, -1)


def labels_to_indices(y):
    y = np.asarray(y).astype(np.int64)
    out = np.zeros_like(y, dtype=np.int64)
    for i, lab in enumerate(y):
        lab = int(lab)
        if lab not in LABEL_TO_INDEX:
            raise ValueError(f"Unexpected label {lab}. Expected labels: {ACTIVITY_IDS}")
        out[i] = LABEL_TO_INDEX[lab]
    return out


def indices_to_labels(y_idx):
    y_idx = np.asarray(y_idx).astype(np.int64)
    return np.array([INDEX_TO_LABEL[int(i)] for i in y_idx], dtype=np.int64)


def to_channels_first(X_native):
    X_native = np.asarray(X_native, dtype=np.float32)
    if X_native.ndim != 3:
        raise ValueError(f"Expected [N,T,C], got {X_native.shape}")
    return np.transpose(X_native, (0, 2, 1)).astype(np.float32)


def resample_time_axis(X_native, target_len):
    X_native = np.asarray(X_native, dtype=np.float32)
    if X_native.ndim != 3:
        raise ValueError(f"Expected [N,T,C], got {X_native.shape}")

    n, old_len, channels = X_native.shape
    if old_len == target_len:
        return X_native.copy().astype(np.float32)
    if old_len < 2:
        raise ValueError(f"Cannot resample time axis with old_len={old_len}")

    old_positions = np.linspace(0.0, old_len - 1, int(target_len), dtype=np.float32)
    left = np.floor(old_positions).astype(np.int64)
    right = np.minimum(left + 1, old_len - 1)
    weight = (old_positions - left).astype(np.float32)

    out = ((1.0 - weight)[None, :, None] * X_native[:, left, :]
           + weight[None, :, None] * X_native[:, right, :])
    return out.astype(np.float32)


def split_slow_to_eda_temp(X_slow):
    X_slow = np.asarray(X_slow, dtype=np.float32)
    if X_slow.ndim != 3 or X_slow.shape[2] != 2:
        raise ValueError(f"Expected SLOW [N,32,2], got {X_slow.shape}")
    return X_slow[:, :, 0:1].astype(np.float32), X_slow[:, :, 1:2].astype(np.float32)


def build_fused_native_view(X_dict, target_len=FUSED_TARGET_LEN):
    X_acc = resample_time_axis(X_dict["acc"], target_len)
    X_bvp = resample_time_axis(X_dict["bvp"], target_len)
    X_eda = resample_time_axis(X_dict["eda"], target_len)
    X_temp = resample_time_axis(X_dict["temp"], target_len)

    lengths = {"acc": len(X_acc), "bvp": len(X_bvp), "eda": len(X_eda), "temp": len(X_temp)}
    if len(set(lengths.values())) != 1:
        raise ValueError(f"Fused view length mismatch: {lengths}")

    return np.concatenate([X_acc, X_bvp, X_eda, X_temp], axis=2).astype(np.float32)


def filter_valid_activities(X_dict, y, subjects):
    keep = np.isin(y, np.array(ACTIVITY_IDS, dtype=np.int64))
    return {k: v[keep] for k, v in X_dict.items()}, y[keep].astype(np.int64), subjects[keep].astype(str)


def filter_by_subjects(X_dict, y, subjects, selected_subjects):
    selected_subjects = set(str(s) for s in selected_subjects)
    keep = np.array([str(s) in selected_subjects for s in subjects], dtype=bool)
    return {k: v[keep] for k, v in X_dict.items()}, y[keep].astype(np.int64), subjects[keep].astype(str)


def sanitize_name(name):
    return str(name).replace(" ", "_").replace("/", "_").replace("\\", "_")



def configure_project_paths(project_root, model_family, synthetic_method, output_root=None, pretrained_results_dir=None):
    global PROJECT_ROOT, PRETRAINED_RESULTS_DIR
    global CURRENT_MODEL_FAMILY, CURRENT_SYNTHETIC_METHOD, CURRENT_SYNTHETIC_METHOD_DISPLAY
    global OUT_DIR, REAL_DIR, SYN_DIR
    global REAL_X_ACC_PATH, REAL_X_BVP_PATH, REAL_X_SLOW_PATH, REAL_Y_PATH, REAL_SUBJECT_PATH
    global SYN_X_ACC_PATH, SYN_X_BVP_PATH, SYN_X_SLOW_PATH, SYN_Y_PATH, SYN_SUBJECT_PATH
    global MODALITY_INFO

    PROJECT_ROOT = Path(project_root)

    if model_family not in MODEL_FAMILY_CONFIGS:
        raise ValueError(f"Unknown model_family={model_family}. Available: {list(MODEL_FAMILY_CONFIGS.keys())}")

    family_cfg = MODEL_FAMILY_CONFIGS[model_family]
    if synthetic_method not in family_cfg["synthetic_methods"]:
        raise ValueError(
            f"Unknown synthetic_method={synthetic_method} for {model_family}. "
            f"Available: {family_cfg['synthetic_methods']}"
        )

    CURRENT_MODEL_FAMILY = model_family
    CURRENT_SYNTHETIC_METHOD = synthetic_method
    CURRENT_SYNTHETIC_METHOD_DISPLAY = family_cfg["method_display_names"].get(synthetic_method, synthetic_method)

    REAL_DIR = PROJECT_ROOT / "data/processed/native_rates"
    SYN_DIR = family_cfg["synthetic_base_dir"] / synthetic_method

    if output_root is None:
        output_root = PROJECT_ROOT / "results/downstream_extended_tsai_aeon"
    else:
        output_root = Path(output_root)

    OUT_DIR = output_root / model_family / synthetic_method
    OUT_DIR.mkdir(parents=True, exist_ok=True)

    if pretrained_results_dir is not None:
        PRETRAINED_RESULTS_DIR = Path(pretrained_results_dir)

    REAL_X_ACC_PATH = REAL_DIR / "all_X_acc_32hz.npy"
    REAL_X_BVP_PATH = REAL_DIR / "all_X_bvp_64hz.npy"
    REAL_X_SLOW_PATH = REAL_DIR / "all_X_slow_4hz.npy"
    REAL_Y_PATH = REAL_DIR / "all_y.npy"
    REAL_SUBJECT_PATH = REAL_DIR / "all_subject.npy"

    SYN_X_ACC_PATH = SYN_DIR / "generated_subjects_X_acc_32hz.npy"
    SYN_X_BVP_PATH = SYN_DIR / "generated_subjects_X_bvp_64hz.npy"
    SYN_X_SLOW_PATH = SYN_DIR / "generated_subjects_X_slow_4hz.npy"
    SYN_Y_PATH = SYN_DIR / "generated_subjects_all_y.npy"
    SYN_SUBJECT_PATH = SYN_DIR / "generated_subjects_all_subject.npy"

    MODALITY_INFO["acc"]["real_path"] = REAL_X_ACC_PATH
    MODALITY_INFO["acc"]["syn_path"] = SYN_X_ACC_PATH
    MODALITY_INFO["bvp"]["real_path"] = REAL_X_BVP_PATH
    MODALITY_INFO["bvp"]["syn_path"] = SYN_X_BVP_PATH
    MODALITY_INFO["slow"]["real_path"] = REAL_X_SLOW_PATH
    MODALITY_INFO["slow"]["syn_path"] = SYN_X_SLOW_PATH

    print("\nConfigured project paths")
    print("  PROJECT_ROOT:", PROJECT_ROOT)
    print("  model_family:", CURRENT_MODEL_FAMILY)
    print("  synthetic_method:", CURRENT_SYNTHETIC_METHOD)
    print("  synthetic_method_display:", CURRENT_SYNTHETIC_METHOD_DISPLAY)
    print("  REAL_DIR:", REAL_DIR)
    print("  SYN_DIR:", SYN_DIR)
    print("  OUT_DIR:", OUT_DIR)
    print("  PRETRAINED_RESULTS_DIR:", PRETRAINED_RESULTS_DIR)


def add_method_columns(df):
    df = df.copy()
    if len(df) == 0:
        return df

    for col, value, pos in [
        ("model_family", CURRENT_MODEL_FAMILY, 0),
        ("synthetic_method", CURRENT_SYNTHETIC_METHOD, 1),
        ("synthetic_method_display_name", CURRENT_SYNTHETIC_METHOD_DISPLAY, 2),
    ]:
        if col in df.columns:
            df[col] = value
        else:
            df.insert(min(pos, len(df.columns)), col, value)

    return df


def result_csv_candidates(framework):
    candidates = [result_csv_path(framework)]

    if PRETRAINED_RESULTS_DIR.exists():
        candidates.append(PRETRAINED_RESULTS_DIR / f"{framework}_native_downstream_results.csv")
        candidates.append(PRETRAINED_RESULTS_DIR / "all_framework_native_results.csv")
        candidates.extend(sorted(PRETRAINED_RESULTS_DIR.glob(f"**/{framework}_native_downstream_results.csv")))
        candidates.extend(sorted(PRETRAINED_RESULTS_DIR.glob("**/all_framework_native_results.csv")))

    unique = []
    seen = set()

    for p in candidates:
        p = Path(p)
        key = str(p.resolve()) if p.exists() else str(p)
        if key not in seen:
            unique.append(p)
            seen.add(key)

    return unique


def activity_csv_candidates(framework):
    candidates = [activity_csv_path(framework)]

    if PRETRAINED_RESULTS_DIR.exists():
        candidates.append(PRETRAINED_RESULTS_DIR / f"{framework}_native_per_activity_results.csv")
        candidates.append(PRETRAINED_RESULTS_DIR / "all_framework_native_per_activity_results.csv")
        candidates.extend(sorted(PRETRAINED_RESULTS_DIR.glob(f"**/{framework}_native_per_activity_results.csv")))
        candidates.extend(sorted(PRETRAINED_RESULTS_DIR.glob("**/all_framework_native_per_activity_results.csv")))

    unique = []
    seen = set()

    for p in candidates:
        p = Path(p)
        key = str(p.resolve()) if p.exists() else str(p)
        if key not in seen:
            unique.append(p)
            seen.add(key)

    return unique


def copy_external_tsai_model_if_needed(model_file):
    model_file = Path(model_file)
    local_dir = OUT_DIR / "tsai_saved_models"
    local_dir.mkdir(parents=True, exist_ok=True)

    local_file = local_dir / model_file.name

    if model_file.exists() and model_file.resolve() != local_file.resolve():
        shutil.copy2(model_file, local_file)
        print("Copied external tsai model into current output folder:", local_file)
        return local_file

    return model_file


def keep_nonflat_cases_for_aeon(X, y, min_std=1e-6, label=""):
    X = np.asarray(X, dtype=np.float32)
    y = np.asarray(y, dtype=np.int64)

    if X.ndim != 3:
        raise ValueError(f"Expected aeon input [N,C,T], got {X.shape}")

    std = X.std(axis=2)
    keep_mask = np.all(std > float(min_std), axis=1)
    dropped = int((~keep_mask).sum())

    if keep_mask.sum() == 0:
        raise ValueError(f"All windows were dropped by aeon low-variation filter for {label}")

    if dropped > 0:
        print(
            f"aeon low-variation filter {label}: "
            f"dropped {dropped}/{len(keep_mask)} windows with at least one channel std <= {min_std}"
        )

    return X[keep_mask], y[keep_mask], dropped


def safe_display(df, max_rows=20):
    try:
        display(df)
    except Exception:
        print(df.head(max_rows))

# =========================
# METRICS AND SAVING
# =========================

def compute_classification_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=np.int64)
    y_pred = np.asarray(y_pred, dtype=np.int64)

    metrics = {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_precision": float(precision_score(y_true, y_pred, labels=ACTIVITY_IDS, average="macro", zero_division=0)),
        "macro_recall": float(recall_score(y_true, y_pred, labels=ACTIVITY_IDS, average="macro", zero_division=0)),
        "macro_f1": float(f1_score(y_true, y_pred, labels=ACTIVITY_IDS, average="macro", zero_division=0)),
        "weighted_precision": float(precision_score(y_true, y_pred, labels=ACTIVITY_IDS, average="weighted", zero_division=0)),
        "weighted_recall": float(recall_score(y_true, y_pred, labels=ACTIVITY_IDS, average="weighted", zero_division=0)),
        "weighted_f1": float(f1_score(y_true, y_pred, labels=ACTIVITY_IDS, average="weighted", zero_division=0)),
    }
    metrics["balanced_accuracy"] = metrics["macro_recall"]
    return metrics


def compute_per_activity_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=np.int64)
    y_pred = np.asarray(y_pred, dtype=np.int64)
    precision, recall, f1, support = precision_recall_fscore_support(
        y_true, y_pred, labels=ACTIVITY_IDS, zero_division=0
    )

    rows = []
    for i, activity in enumerate(ACTIVITY_IDS):
        true_mask = y_true == activity
        correct = int(np.sum(true_mask & (y_pred == activity)))
        total = int(np.sum(true_mask))
        rows.append({
            "activity_label": int(activity),
            "precision": float(precision[i]),
            "recall": float(recall[i]),
            "f1": float(f1[i]),
            "support": int(support[i]),
            "correct_true_activity_windows": correct,
            "total_true_activity_windows": total,
            "true_activity_window_accuracy": float(correct / total) if total > 0 else np.nan,
        })
    return pd.DataFrame(rows)


def plot_confusion_matrix_image(cm, labels, title, out_path, normalize=False):
    cm_to_plot = cm.astype(np.float64)
    if normalize:
        row_sum = cm_to_plot.sum(axis=1, keepdims=True)
        cm_to_plot = np.divide(cm_to_plot, row_sum, out=np.zeros_like(cm_to_plot), where=row_sum != 0)

    fig, ax = plt.subplots(figsize=(8, 7))
    im = ax.imshow(cm_to_plot, cmap="Blues")
    ax.figure.colorbar(im, ax=ax)
    ax.set_title(title)
    ax.set_xlabel("Predicted activity")
    ax.set_ylabel("True activity")
    ax.set_xticks(np.arange(len(labels)))
    ax.set_yticks(np.arange(len(labels)))
    ax.set_xticklabels(labels)
    ax.set_yticklabels(labels)

    threshold = cm_to_plot.max() / 2.0 if cm_to_plot.size and cm_to_plot.max() > 0 else 0.0
    for i in range(cm_to_plot.shape[0]):
        for j in range(cm_to_plot.shape[1]):
            text = f"{cm_to_plot[i, j]:.2f}" if normalize else str(int(cm[i, j]))
            ax.text(j, i, text, ha="center", va="center", color="white" if cm_to_plot[i, j] > threshold else "black")

    fig.tight_layout()
    fig.savefig(out_path, dpi=200, bbox_inches="tight")
    plt.close(fig)


def save_predictions_cm_and_activity_report(framework, model, experiment, modality, native_hz, channels, y_true, y_pred):
    safe_name = sanitize_name(f"{framework}_{experiment}_{modality}")
    y_true = np.asarray(y_true, dtype=np.int64)
    y_pred = np.asarray(y_pred, dtype=np.int64)

    pred_path = OUT_DIR / f"{safe_name}_predictions.csv"
    pd.DataFrame({"y_true": y_true, "y_pred": y_pred, "correct": y_true == y_pred}).to_csv(pred_path, index=False)

    cm = confusion_matrix(y_true, y_pred, labels=ACTIVITY_IDS)
    cm_df = pd.DataFrame(cm, index=[f"true_{a}" for a in ACTIVITY_IDS], columns=[f"pred_{a}" for a in ACTIVITY_IDS])
    cm_csv_path = OUT_DIR / f"{safe_name}_confusion_matrix.csv"
    cm_df.to_csv(cm_csv_path)

    cm_count_png = OUT_DIR / f"{safe_name}_confusion_matrix_counts.png"
    cm_norm_png = OUT_DIR / f"{safe_name}_confusion_matrix_normalized.png"
    plot_confusion_matrix_image(cm, ACTIVITY_IDS, f"{framework} {experiment} {modality} confusion matrix", cm_count_png, normalize=False)
    plot_confusion_matrix_image(cm, ACTIVITY_IDS, f"{framework} {experiment} {modality} normalized confusion matrix", cm_norm_png, normalize=True)

    activity_df = compute_per_activity_metrics(y_true, y_pred)
    activity_df.insert(0, "framework", framework)
    activity_df.insert(1, "model", model)
    activity_df.insert(2, "experiment", experiment)
    activity_df.insert(3, "native_modality", modality)
    activity_df.insert(4, "native_hz", native_hz)
    activity_df.insert(5, "channels", ",".join(channels))

    activity_path = OUT_DIR / f"{safe_name}_per_activity_metrics.csv"
    activity_df.to_csv(activity_path, index=False)

    print("Saved:", pred_path)
    print("Saved:", cm_csv_path)
    print("Saved:", cm_count_png)
    print("Saved:", cm_norm_png)
    print("Saved:", activity_path)
    return activity_df

# =========================
# DATASET VIEWS
# =========================

def load_native_data(load_synthetic):
    require_file(REAL_Y_PATH)
    require_file(REAL_SUBJECT_PATH)

    real_y = np.load(REAL_Y_PATH).astype(np.int64)
    real_subjects = np.load(REAL_SUBJECT_PATH, allow_pickle=True).astype(str)
    if len(real_y) != len(real_subjects):
        raise ValueError(f"Real y/subject mismatch: {len(real_y)} vs {len(real_subjects)}")

    real_X = {}
    for modality_name, info in MODALITY_INFO.items():
        require_file(info["real_path"])
        Xr = np.load(info["real_path"]).astype(np.float32)
        if Xr.ndim != 3 or Xr.shape[1:] != info["expected_shape_tail"]:
            raise ValueError(f"Real {modality_name}: expected [N,{info['expected_shape_tail']}], got {Xr.shape}")
        if len(Xr) != len(real_y):
            raise ValueError(f"Real {modality_name}/y mismatch: {len(Xr)} vs {len(real_y)}")
        real_X[modality_name] = Xr

    real_X, real_y, real_subjects = filter_valid_activities(real_X, real_y, real_subjects)
    print("\nLoaded real data:")
    print("  y:", real_y.shape)
    print("  subjects:", real_subjects.shape)
    for k, v in real_X.items():
        print(f"  {k}: {v.shape}")

    if not load_synthetic:
        print("\nSynthetic data skipped because RUN_REAL_TO_REAL_ONLY=True.")
        return real_X, real_y, real_subjects, None, None, None

    require_file(SYN_Y_PATH)
    require_file(SYN_SUBJECT_PATH)
    syn_y = np.load(SYN_Y_PATH).astype(np.int64)
    syn_subjects = np.load(SYN_SUBJECT_PATH, allow_pickle=True).astype(str)
    if len(syn_y) != len(syn_subjects):
        raise ValueError(f"Synthetic y/subject mismatch: {len(syn_y)} vs {len(syn_subjects)}")

    syn_X = {}
    for modality_name, info in MODALITY_INFO.items():
        require_file(info["syn_path"])
        Xs = np.load(info["syn_path"]).astype(np.float32)
        if Xs.ndim != 3 or Xs.shape[1:] != info["expected_shape_tail"]:
            raise ValueError(f"Synthetic {modality_name}: expected [N,{info['expected_shape_tail']}], got {Xs.shape}")
        if len(Xs) != len(syn_y):
            raise ValueError(f"Synthetic {modality_name}/y mismatch: {len(Xs)} vs {len(syn_y)}")
        syn_X[modality_name] = Xs

    syn_X, syn_y, syn_subjects = filter_valid_activities(syn_X, syn_y, syn_subjects)
    print("\nLoaded synthetic data:")
    print("  y:", syn_y.shape)
    print("  subjects:", syn_subjects.shape)
    for k, v in syn_X.items():
        print(f"  {k}: {v.shape}")

    return real_X, real_y, real_subjects, syn_X, syn_y, syn_subjects


def validate_fixed_split(real_subjects):
    available = sorted(np.unique(real_subjects.astype(str)), key=subject_sort_key)
    missing_train = sorted(set(TRAIN_SUBJECTS) - set(available), key=subject_sort_key)
    missing_val = sorted(set(VAL_SUBJECTS) - set(available), key=subject_sort_key)
    missing_test = sorted(set(TEST_SUBJECTS) - set(available), key=subject_sort_key)

    if missing_train or missing_val or missing_test:
        raise ValueError(
            "Fixed split subjects missing from real data.\n"
            f"Missing train: {missing_train}\nMissing val: {missing_val}\nMissing test: {missing_test}\nAvailable: {available}"
        )
    if set(TRAIN_SUBJECTS) & set(VAL_SUBJECTS) or set(TRAIN_SUBJECTS) & set(TEST_SUBJECTS) or set(VAL_SUBJECTS) & set(TEST_SUBJECTS):
        raise ValueError("Train/val/test subject split overlaps.")

    print("\nFixed split OK:")
    print("  Train:", TRAIN_SUBJECTS)
    print("  Val:  ", VAL_SUBJECTS)
    print("  Test: ", TEST_SUBJECTS)


def make_native_dataset_views(real_X_all, real_y_all, real_subjects_all, syn_X_all=None, syn_y_all=None, syn_subjects_all=None):
    validate_fixed_split(real_subjects_all)
    use_synthetic = syn_X_all is not None and syn_y_all is not None and syn_subjects_all is not None

    real_train_X, real_train_y, _ = filter_by_subjects(real_X_all, real_y_all, real_subjects_all, TRAIN_SUBJECTS)
    real_val_X, real_val_y, _ = filter_by_subjects(real_X_all, real_y_all, real_subjects_all, VAL_SUBJECTS)
    real_test_X, real_test_y, _ = filter_by_subjects(real_X_all, real_y_all, real_subjects_all, TEST_SUBJECTS)

    syn_test_subjects = []
    syn_test_X = None
    syn_test_y = None
    if use_synthetic:
        unique_syn_subjects = sorted(np.unique(syn_subjects_all.astype(str)), key=subject_sort_key)
        if len(unique_syn_subjects) < SYN_TEST_N_SUBJECTS:
            raise ValueError(f"Need at least {SYN_TEST_N_SUBJECTS} synthetic subjects, found {len(unique_syn_subjects)}")
        syn_test_subjects = unique_syn_subjects[:SYN_TEST_N_SUBJECTS]
        syn_test_X, syn_test_y, _ = filter_by_subjects(syn_X_all, syn_y_all, syn_subjects_all, syn_test_subjects)
        print("\nSynthetic test subjects:", syn_test_subjects)

    real_train_eda, real_train_temp = split_slow_to_eda_temp(real_train_X["slow"])
    real_val_eda, real_val_temp = split_slow_to_eda_temp(real_val_X["slow"])
    real_test_eda, real_test_temp = split_slow_to_eda_temp(real_test_X["slow"])

    real_train_eval = {"acc": real_train_X["acc"], "bvp": real_train_X["bvp"], "eda": real_train_eda, "temp": real_train_temp}
    real_val_eval = {"acc": real_val_X["acc"], "bvp": real_val_X["bvp"], "eda": real_val_eda, "temp": real_val_temp}
    real_test_eval = {"acc": real_test_X["acc"], "bvp": real_test_X["bvp"], "eda": real_test_eda, "temp": real_test_temp}

    syn_eval = None
    syn_test_eval = None
    if use_synthetic:
        syn_eda, syn_temp = split_slow_to_eda_temp(syn_X_all["slow"])
        syn_test_eda, syn_test_temp = split_slow_to_eda_temp(syn_test_X["slow"])
        syn_eval = {"acc": syn_X_all["acc"], "bvp": syn_X_all["bvp"], "eda": syn_eda, "temp": syn_temp}
        syn_test_eval = {"acc": syn_test_X["acc"], "bvp": syn_test_X["bvp"], "eda": syn_test_eda, "temp": syn_test_temp}

    info = {
        "acc": {"native_hz": 32, "channels": ["ACC_x", "ACC_y", "ACC_z"], "description": "ACC only"},
        "bvp": {"native_hz": 64, "channels": ["BVP"], "description": "BVP only"},
        "eda": {"native_hz": 4, "channels": ["EDA"], "description": "EDA only"},
        "temp": {"native_hz": 4, "channels": ["TEMP"], "description": "TEMP only"},
    }

    views = {}
    for modality in ["acc", "bvp", "eda", "temp"]:
        item = {
            "real_train_X_native": real_train_eval[modality],
            "real_val_X_native": real_val_eval[modality],
            "real_test_X_native": real_test_eval[modality],
            "real_train_y": real_train_y,
            "real_val_y": real_val_y,
            "real_test_y": real_test_y,
            "native_hz": info[modality]["native_hz"],
            "channels": info[modality]["channels"],
            "description": info[modality]["description"],
        }
        if use_synthetic:
            item.update({
                "syn_X_native": syn_eval[modality],
                "syn_test_X_native": syn_test_eval[modality],
                "syn_y": syn_y_all,
                "syn_test_y": syn_test_y,
                "syn_test_subjects": syn_test_subjects,
            })
        views[modality] = item

    fused_item = {
        "real_train_X_native": build_fused_native_view(real_train_eval),
        "real_val_X_native": build_fused_native_view(real_val_eval),
        "real_test_X_native": build_fused_native_view(real_test_eval),
        "real_train_y": real_train_y,
        "real_val_y": real_val_y,
        "real_test_y": real_test_y,
        "native_hz": FUSED_NATIVE_HZ,
        "channels": FUSED_CHANNEL_NAMES,
        "description": "ACC+BVP+EDA+TEMP fused",
    }
    if use_synthetic:
        fused_item.update({
            "syn_X_native": build_fused_native_view(syn_eval),
            "syn_test_X_native": build_fused_native_view(syn_test_eval),
            "syn_y": syn_y_all,
            "syn_test_y": syn_test_y,
            "syn_test_subjects": syn_test_subjects,
        })
    views[FUSED_MODALITY_NAME] = fused_item

    for name, view in views.items():
        msg = f"  View {name:6s}: train={view['real_train_X_native'].shape}, val={view['real_val_X_native'].shape}, test={view['real_test_X_native'].shape}"
        if use_synthetic:
            msg += f", syn_all={view['syn_X_native'].shape}, syn_test3={view['syn_test_X_native'].shape}"
        print(msg)

    return views


def save_shape_report(views):
    rows = []
    for name, view in views.items():
        row = {
            "native_view": name,
            "native_hz": view["native_hz"],
            "channels": ",".join(view["channels"]),
            "native_real_train_shape_N_T_C": list(view["real_train_X_native"].shape),
            "native_real_val_shape_N_T_C": list(view["real_val_X_native"].shape),
            "native_real_test_shape_N_T_C": list(view["real_test_X_native"].shape),
            "framework_real_train_shape_N_C_T": list(to_channels_first(view["real_train_X_native"]).shape),
            "framework_real_val_shape_N_C_T": list(to_channels_first(view["real_val_X_native"]).shape),
            "framework_real_test_shape_N_C_T": list(to_channels_first(view["real_test_X_native"]).shape),
        }
        if "syn_X_native" in view:
            row.update({
                "native_syn_all_shape_N_T_C": list(view["syn_X_native"].shape),
                "native_syn_test3_shape_N_T_C": list(view["syn_test_X_native"].shape),
                "syn_test_subjects": ",".join(view.get("syn_test_subjects", [])),
                "framework_syn_all_shape_N_C_T": list(to_channels_first(view["syn_X_native"]).shape),
                "framework_syn_test3_shape_N_C_T": list(to_channels_first(view["syn_test_X_native"]).shape),
            })
        rows.append(row)

    df = pd.DataFrame(rows)
    out_path = OUT_DIR / "native_input_shape_report.csv"
    df.to_csv(out_path, index=False)
    print("\nSaved shape report:", out_path)
    return df

# =========================
# EXPERIMENTS AND SAVED ROWS
# =========================

def get_experiments_for_view(view):
    real_to_real = {
        "name": "real_to_real",
        "train_X": view["real_train_X_native"],
        "train_y": view["real_train_y"],
        "val_X": view["real_val_X_native"],
        "val_y": view["real_val_y"],
        "test_X": view["real_test_X_native"],
        "test_y": view["real_test_y"],
    }

    if RUN_REAL_TO_REAL_ONLY:
        return [real_to_real]

    missing_syn_keys = [k for k in ["syn_X_native", "syn_test_X_native", "syn_y", "syn_test_y"] if k not in view]
    if missing_syn_keys:
        raise ValueError(f"Full evaluation needs synthetic data. Missing keys: {missing_syn_keys}")

    return [
        real_to_real,
        {
            "name": "real_to_synthetic",
            "train_X": view["real_train_X_native"],
            "train_y": view["real_train_y"],
            "val_X": view["real_val_X_native"],
            "val_y": view["real_val_y"],
            "test_X": view["syn_test_X_native"],
            "test_y": view["syn_test_y"],
        },
        {
            "name": "synthetic_to_real",
            "train_X": view["syn_X_native"],
            "train_y": view["syn_y"],
            "val_X": view["real_val_X_native"],
            "val_y": view["real_val_y"],
            "test_X": view["real_test_X_native"],
            "test_y": view["real_test_y"],
        },
        {
            "name": "real_plus_synthetic_to_real",
            "train_X": np.concatenate([view["real_train_X_native"], view["syn_X_native"]], axis=0),
            "train_y": np.concatenate([view["real_train_y"], view["syn_y"]], axis=0),
            "val_X": view["real_val_X_native"],
            "val_y": view["real_val_y"],
            "test_X": view["real_test_X_native"],
            "test_y": view["real_test_y"],
        },
    ]


def result_csv_path(framework):
    return OUT_DIR / f"{framework}_native_downstream_results.csv"


def activity_csv_path(framework):
    return OUT_DIR / f"{framework}_native_per_activity_results.csv"


def get_existing_result_row(framework, experiment, modality, model=None):
    for path in result_csv_candidates(framework):
        if not Path(path).exists():
            continue
        try:
            df = pd.read_csv(path)
        except Exception:
            continue

        required = {"framework", "experiment", "native_modality"}
        if not required.issubset(df.columns):
            continue

        mask = (
            (df["framework"].astype(str) == framework)
            & (df["experiment"].astype(str) == experiment)
            & (df["native_modality"].astype(str) == modality)
        )
        if model is not None and "model" in df.columns:
            mask &= df["model"].astype(str) == str(model)

        if mask.any():
            row = df.loc[mask].iloc[-1].to_dict()
            row["_loaded_from_result_csv"] = str(path)
            return row

    return None


def get_existing_activity_rows(framework, experiment, modality, model=None):
    for path in activity_csv_candidates(framework):
        if not Path(path).exists():
            continue
        try:
            df = pd.read_csv(path)
        except Exception:
            continue

        required = {"framework", "experiment", "native_modality"}
        if not required.issubset(df.columns):
            continue

        mask = (
            (df["framework"].astype(str) == framework)
            & (df["experiment"].astype(str) == experiment)
            & (df["native_modality"].astype(str) == modality)
        )
        if model is not None and "model" in df.columns:
            mask &= df["model"].astype(str) == str(model)

        if mask.any():
            return df.loc[mask].copy()

    return pd.DataFrame()

# =========================
# TSAI
# =========================

def import_tsai_stuff():
    try:
        import torch
        import tsai.all as tsai_all
        from tsai.all import get_ts_dls, TSClassification, TSStandardize, ts_learner
        try:
            from fastai.metrics import accuracy
        except Exception:
            from tsai.all import accuracy
        from fastai.callback.tracker import SaveModelCallback
    except Exception as e:
        raise ImportError(f"Could not import tsai/fastai: {repr(e)}")

    if not hasattr(tsai_all, TSAI_ARCH):
        raise ValueError(f"TSAI_ARCH='{TSAI_ARCH}' was not found in tsai.all")

    return {
        "torch": torch,
        "get_ts_dls": get_ts_dls,
        "TSClassification": TSClassification,
        "TSStandardize": TSStandardize,
        "ts_learner": ts_learner,
        "accuracy": accuracy,
        "SaveModelCallback": SaveModelCallback,
        "arch_obj": getattr(tsai_all, TSAI_ARCH),
    }


def extract_tsai_pred_indices(pred_decoded):
    pred = pred_decoded.detach().cpu().numpy() if hasattr(pred_decoded, "detach") else np.asarray(pred_decoded)
    pred = np.asarray(pred).reshape(-1)
    return np.array([int(x.item() if hasattr(x, "item") else x) for x in pred], dtype=np.int64)


def get_tsai_model_file(experiment, modality):
    stem = sanitize_name(f"best_{TSAI_ARCH}_{experiment}_{modality}")
    return OUT_DIR / "tsai_saved_models" / f"{stem}.pth", stem


def find_tsai_real_model_file(modality):
    candidates = []
    row = get_existing_result_row("tsai", "real_to_real", modality, model=TSAI_ARCH)
    if row is not None:
        saved = row.get("saved_best_model_file", "")
        if str(saved).lower() not in {"", "nan", "none"}:
            candidates.append(Path(saved))

    candidates.append(get_tsai_model_file("real_to_real", modality)[0])

    if PRETRAINED_RESULTS_DIR.exists():
        candidates.extend(sorted(PRETRAINED_RESULTS_DIR.glob(f"**/*{sanitize_name(TSAI_ARCH)}*real_to_real*{sanitize_name(modality)}*.pth")))
        candidates.extend(sorted(PRETRAINED_RESULTS_DIR.glob(f"**/*real_to_real*{sanitize_name(modality)}*.pth")))

    unique = []
    seen = set()
    for p in candidates:
        p = Path(p)
        key = str(p.resolve()) if p.exists() else str(p)
        if key not in seen:
            unique.append(p)
            seen.add(key)

    for p in unique:
        if Path(p).exists():
            return Path(p)

    raise FileNotFoundError(
        f"Missing tsai Real->Real model for modality '{modality}'. "
        f"Looked in current OUT_DIR and PRETRAINED_RESULTS_DIR={PRETRAINED_RESULTS_DIR}."
    )


def build_tsai_learner(exp, tsai_obj):
    X_train = to_channels_first(exp["train_X"])
    X_val = to_channels_first(exp["val_X"])
    X_test = to_channels_first(exp["test_X"])

    y_train_idx = labels_to_indices(exp["train_y"])
    y_val_idx = labels_to_indices(exp["val_y"])
    y_test_idx = labels_to_indices(exp["test_y"])

    X_trainval = np.concatenate([X_train, X_val], axis=0).astype(np.float32)
    y_trainval = np.concatenate([y_train_idx, y_val_idx], axis=0).astype(np.int64)
    splits = (np.arange(0, len(X_train)), np.arange(len(X_train), len(X_trainval)))

    dls = tsai_obj["get_ts_dls"](
        X_trainval,
        y_trainval,
        splits=splits,
        tfms=[None, tsai_obj["TSClassification"]()],
        batch_tfms=tsai_obj["TSStandardize"](),
        bs=BATCH_SIZE,
    )
    learn = tsai_obj["ts_learner"](dls, tsai_obj["arch_obj"], metrics=tsai_obj["accuracy"])
    learn.path = OUT_DIR
    learn.model_dir = "tsai_saved_models"

    return {
        "learn": learn,
        "dls": dls,
        "X_train": X_train,
        "X_test": X_test,
        "y_test_idx": y_test_idx,
        "y_true": indices_to_labels(y_test_idx),
    }


def get_best_tsai_epoch_and_score(learn, monitor="accuracy"):
    values = list(getattr(learn.recorder, "values", []))
    if not values:
        return None, None
    metric_names = list(getattr(learn.recorder, "metric_names", []))
    value_names = metric_names[1:-1] if len(metric_names) >= 2 and len(metric_names) == len(values[0]) + 2 else metric_names
    if monitor not in value_names:
        return None, None
    monitor_values = np.array([float(row[value_names.index(monitor)]) for row in values], dtype=np.float64)
    if len(monitor_values) == 0 or np.all(np.isnan(monitor_values)):
        return None, None
    return int(np.nanargmax(monitor_values)) + 1, float(np.nanmax(monitor_values))


def predict_tsai(learn, X_test, y_test_idx):
    pred_output = learn.get_X_preds(X_test, y_test_idx, bs=BATCH_SIZE, with_decoded=True)
    return indices_to_labels(extract_tsai_pred_indices(pred_output[-1]))


def make_tsai_row(exp, view, modality, X_train, X_test, metrics, epochs, best_epoch, best_val_accuracy,
                  model_file, used_real_model=False):
    return {
        "framework": "tsai",
        "model": TSAI_ARCH,
        "experiment": exp["name"],
        "native_modality": modality,
        "native_hz": view["native_hz"],
        "channels": ",".join(view["channels"]),
        "description": view.get("description", ""),
        "input_shape_train_native_N_T_C": list(exp["train_X"].shape),
        "input_shape_test_native_N_T_C": list(exp["test_X"].shape),
        "input_shape_train_framework_N_C_T": list(X_train.shape),
        "input_shape_test_framework_N_C_T": list(X_test.shape),
        "epochs": epochs,
        "best_epoch_by_val_accuracy": best_epoch,
        "best_val_accuracy": best_val_accuracy,
        "saved_best_model_file": str(model_file),
        "used_saved_real_to_real_model": bool(used_real_model),
        "training_reused_from_experiment": "real_to_real" if used_real_model else "",
        "batch_size": BATCH_SIZE,
        "lr": LR_TSAI,
        **metrics,
    }


def run_tsai(views, modalities):
    try:
        tsai_obj = import_tsai_stuff()
    except Exception as e:
        print("\n[tsai] Skipping tsai.")
        print("Error:", repr(e))
        return pd.DataFrame(), pd.DataFrame()

    rows = []
    activity_rows = []

    for modality in modalities:
        if modality not in views:
            print(f"[tsai] Skipping unknown modality: {modality}")
            continue

        view = views[modality]
        for exp in get_experiments_for_view(view):
            print("\n" + "=" * 80)
            print(f"[tsai] {exp['name']} | {modality}")
            print("=" * 80)

            if exp["name"] == "real_to_real" and not RUN_REAL_TO_REAL_ONLY:
                existing = get_existing_result_row("tsai", "real_to_real", modality, model=TSAI_ARCH)
                if existing is not None:
                    print("Reusing saved tsai Real->Real metrics.")
                    rows.append(existing)
                    existing_activity = get_existing_activity_rows("tsai", "real_to_real", modality, model=TSAI_ARCH)
                    if len(existing_activity) > 0:
                        activity_rows.append(existing_activity)
                    continue

            set_seed(SEED)
            prepared = None
            try:
                prepared = build_tsai_learner(exp, tsai_obj)
                learn = prepared["learn"]

                if exp["name"] == "real_to_synthetic" and USE_REAL_MODELS:
                    model_file = find_tsai_real_model_file(modality)
                    model_file = copy_external_tsai_model_if_needed(model_file)
                    print("Loading tsai Real->Real model:", model_file)
                    learn.load(model_file.stem)
                    best_epoch = None
                    best_val_accuracy = None
                    source = get_existing_result_row("tsai", "real_to_real", modality, model=TSAI_ARCH)
                    if source is not None:
                        best_epoch = source.get("best_epoch_by_val_accuracy", None)
                        best_val_accuracy = source.get("best_val_accuracy", None)
                    epochs = 0
                    used_real_model = True
                else:
                    model_file, model_stem = get_tsai_model_file(exp["name"], modality)
                    save_best_cb = tsai_obj["SaveModelCallback"](monitor="accuracy", comp=np.greater, fname=model_stem)
                    learn.fit_one_cycle(EPOCHS_TSAI, LR_TSAI, cbs=[save_best_cb])
                    best_epoch, best_val_accuracy = get_best_tsai_epoch_and_score(learn, monitor="accuracy")
                    learn.load(model_stem)
                    epochs = EPOCHS_TSAI
                    used_real_model = False

                y_true = prepared["y_true"]
                y_pred = predict_tsai(learn, prepared["X_test"], prepared["y_test_idx"])
                metrics = compute_classification_metrics(y_true, y_pred)

                activity_df = save_predictions_cm_and_activity_report(
                    "tsai", TSAI_ARCH, exp["name"], modality, view["native_hz"], view["channels"], y_true, y_pred
                )
                activity_rows.append(activity_df)

                row = make_tsai_row(
                    exp, view, modality, prepared["X_train"], prepared["X_test"], metrics,
                    epochs, best_epoch, best_val_accuracy, model_file, used_real_model=used_real_model
                )
                rows.append(row)
                print(json.dumps(row, indent=2))

            except Exception as e:
                warnings.warn(f"[tsai] Failed {exp['name']} | {modality}: {repr(e)}")
                rows.append({"framework": "tsai", "model": TSAI_ARCH, "experiment": exp["name"], "native_modality": modality, "error": repr(e)})

            finally:
                try:
                    del prepared
                    if tsai_obj["torch"].cuda.is_available():
                        tsai_obj["torch"].cuda.empty_cache()
                except Exception:
                    pass

    df = pd.DataFrame(rows)
    df.to_csv(result_csv_path("tsai"), index=False)
    activity_df_all = pd.concat(activity_rows, ignore_index=True) if activity_rows else pd.DataFrame()
    activity_df_all.to_csv(activity_csv_path("tsai"), index=False)
    print("\n[tsai] Saved:", result_csv_path("tsai"))
    print("[tsai] Saved:", activity_csv_path("tsai"))
    return df, activity_df_all

# =========================
# AEON
# =========================

def import_aeon_classifier_class():
    try:
        from aeon.classification.convolution_based import MiniRocketClassifier
        return MiniRocketClassifier
    except Exception:
        try:
            from aeon.classification.convolution_based import RocketClassifier
            return RocketClassifier
        except Exception as e:
            raise ImportError(f"Could not import MiniRocketClassifier or RocketClassifier: {repr(e)}")


def make_aeon_classifier():
    Classifier = import_aeon_classifier_class()
    params = inspect.signature(Classifier).parameters
    kwargs = {}
    if "n_kernels" in params:
        kwargs["n_kernels"] = AEON_N_KERNELS
    if "num_kernels" in params:
        kwargs["num_kernels"] = AEON_N_KERNELS
    if "n_jobs" in params:
        kwargs["n_jobs"] = AEON_N_JOBS
    if "random_state" in params:
        kwargs["random_state"] = SEED
    return Classifier(**kwargs)


def get_aeon_model_file(model_name, experiment, modality):
    return OUT_DIR / "aeon_saved_models" / f"{sanitize_name(model_name)}_{sanitize_name(experiment)}_{sanitize_name(modality)}.joblib"


def find_aeon_real_model_file(modality):
    candidates = []
    row = get_existing_result_row("aeon", "real_to_real", modality)
    if row is not None:
        saved = row.get("saved_model_file", "")
        if str(saved).lower() not in {"", "nan", "none"}:
            candidates.append(Path(saved))

    model_dir = OUT_DIR / "aeon_saved_models"
    if model_dir.exists():
        candidates.extend(sorted(model_dir.glob(f"*_real_to_real_{sanitize_name(modality)}.joblib")))

    if PRETRAINED_RESULTS_DIR.exists():
        candidates.extend(sorted(PRETRAINED_RESULTS_DIR.glob(f"**/*_real_to_real_{sanitize_name(modality)}.joblib")))
        candidates.extend(sorted(PRETRAINED_RESULTS_DIR.glob(f"**/*real_to_real*{sanitize_name(modality)}*.joblib")))

    unique = []
    seen = set()
    for p in candidates:
        p = Path(p)
        key = str(p.resolve()) if p.exists() else str(p)
        if key not in seen:
            unique.append(p)
            seen.add(key)

    for p in unique:
        if Path(p).exists():
            return Path(p)

    raise FileNotFoundError(
        f"Missing aeon Real->Real model for modality '{modality}'. "
        f"Looked in current OUT_DIR and PRETRAINED_RESULTS_DIR={PRETRAINED_RESULTS_DIR}."
    )


def save_aeon_model(clf, experiment, modality):
    model_dir = OUT_DIR / "aeon_saved_models"
    model_dir.mkdir(parents=True, exist_ok=True)
    model_path = get_aeon_model_file(clf.__class__.__name__, experiment, modality)
    joblib.dump(clf, model_path)
    print("Saved aeon model:", model_path)
    return str(model_path)


def make_aeon_row(exp, view, modality, model_name, X_train, X_test, metrics, saved_model_file,
                  used_real_model=False):
    return {
        "framework": "aeon",
        "model": model_name,
        "experiment": exp["name"],
        "native_modality": modality,
        "native_hz": view["native_hz"],
        "channels": ",".join(view["channels"]),
        "description": view.get("description", ""),
        "input_shape_train_native_N_T_C": list(exp["train_X"].shape),
        "input_shape_test_native_N_T_C": list(exp["test_X"].shape),
        "input_shape_train_framework_N_C_T": list(X_train.shape),
        "input_shape_test_framework_N_C_T": list(X_test.shape),
        "n_kernels": AEON_N_KERNELS,
        "n_jobs": AEON_N_JOBS,
        "saved_model_file": str(saved_model_file),
        "used_saved_real_to_real_model": bool(used_real_model),
        "training_reused_from_experiment": "real_to_real" if used_real_model else "",
        **metrics,
    }


def run_aeon(views, modalities):
    try:
        import aeon  # noqa: F401
    except Exception as e:
        print("\n[aeon] Skipping aeon.")
        print("Error:", repr(e))
        return pd.DataFrame(), pd.DataFrame()

    rows = []
    activity_rows = []

    for modality in modalities:
        if modality not in views:
            print(f"[aeon] Skipping unknown modality: {modality}")
            continue

        view = views[modality]
        for exp in get_experiments_for_view(view):
            print("\n" + "=" * 80)
            print(f"[aeon] {exp['name']} | {modality}")
            print("=" * 80)

            if exp["name"] == "real_to_real" and not RUN_REAL_TO_REAL_ONLY:
                existing = get_existing_result_row("aeon", "real_to_real", modality)
                if existing is not None:
                    print("Reusing saved aeon Real->Real metrics.")
                    rows.append(existing)
                    existing_activity = get_existing_activity_rows("aeon", "real_to_real", modality)
                    if len(existing_activity) > 0:
                        activity_rows.append(existing_activity)
                    continue

            X_train = to_channels_first(exp["train_X"])
            X_test = to_channels_first(exp["test_X"])
            y_train = exp["train_y"].astype(np.int64)
            y_test = exp["test_y"].astype(np.int64)

            train_windows_original_before_aeon_filter = int(len(y_train))
            test_windows_original_before_aeon_filter = int(len(y_test))
            train_windows_dropped_by_aeon_filter = 0
            test_windows_dropped_by_aeon_filter = 0

            if AEON_FIX_LOW_VARIATION:
                X_test, y_test, test_windows_dropped_by_aeon_filter = keep_nonflat_cases_for_aeon(
                    X_test,
                    y_test,
                    min_std=AEON_MIN_STD,
                    label=f"test {exp['name']} {modality}",
                )

                if not (exp["name"] == "real_to_synthetic" and USE_REAL_MODELS):
                    X_train, y_train, train_windows_dropped_by_aeon_filter = keep_nonflat_cases_for_aeon(
                        X_train,
                        y_train,
                        min_std=AEON_MIN_STD,
                        label=f"train {exp['name']} {modality}",
                    )

            try:
                if exp["name"] == "real_to_synthetic" and USE_REAL_MODELS:
                    model_file = find_aeon_real_model_file(modality)
                    print("Loading aeon Real->Real model:", model_file)
                    clf = joblib.load(model_file)
                    saved_model_file = str(model_file)
                    used_real_model = True
                else:
                    set_seed(SEED)
                    clf = make_aeon_classifier()
                    print("Aeon model:", clf.__class__.__name__)
                    print("Train shape:", X_train.shape)
                    print("Test shape: ", X_test.shape)
                    clf.fit(X_train, y_train)
                    saved_model_file = save_aeon_model(clf, exp["name"], modality)
                    used_real_model = False

                y_pred = clf.predict(X_test).astype(np.int64)
                metrics = compute_classification_metrics(y_test, y_pred)

                activity_df = save_predictions_cm_and_activity_report(
                    "aeon", clf.__class__.__name__, exp["name"], modality, view["native_hz"], view["channels"], y_test, y_pred
                )
                activity_rows.append(activity_df)

                row = make_aeon_row(
                    exp, view, modality, clf.__class__.__name__, X_train, X_test,
                    metrics, saved_model_file, used_real_model=used_real_model
                )
                row.update({
                    "train_windows_original_before_aeon_filter": train_windows_original_before_aeon_filter,
                    "train_windows_dropped_by_aeon_filter": train_windows_dropped_by_aeon_filter,
                    "test_windows_original_before_aeon_filter": test_windows_original_before_aeon_filter,
                    "test_windows_dropped_by_aeon_filter": test_windows_dropped_by_aeon_filter,
                    "aeon_fix_low_variation": AEON_FIX_LOW_VARIATION,
                    "aeon_min_std": AEON_MIN_STD,
                })
                rows.append(row)
                print(json.dumps(row, indent=2))

            except Exception as e:
                warnings.warn(f"[aeon] Failed {exp['name']} | {modality}: {repr(e)}")
                rows.append({"framework": "aeon", "experiment": exp["name"], "native_modality": modality, "error": repr(e)})

    df = pd.DataFrame(rows)
    df.to_csv(result_csv_path("aeon"), index=False)
    activity_df_all = pd.concat(activity_rows, ignore_index=True) if activity_rows else pd.DataFrame()
    activity_df_all.to_csv(activity_csv_path("aeon"), index=False)
    print("\n[aeon] Saved:", result_csv_path("aeon"))
    print("[aeon] Saved:", activity_csv_path("aeon"))
    return df, activity_df_all

# =========================
# RUN AND COVERAGE
# =========================

def expected_experiments():
    if RUN_REAL_TO_REAL_ONLY:
        return ["real_to_real"]
    return ["real_to_real", "real_to_synthetic", "synthetic_to_real", "real_plus_synthetic_to_real"]


def save_coverage_report(final_results_df):
    expected = pd.DataFrame(
        list(itertools.product(FRAMEWORKS, expected_experiments(), MODALITIES)),
        columns=["framework", "experiment", "native_modality"],
    )

    if len(final_results_df) > 0:
        df = final_results_df.copy()
        if "error" not in df.columns:
            df["error"] = ""
        df["has_error"] = df["error"].fillna("").astype(str) != ""
        actual = (
            df.groupby(["framework", "experiment", "native_modality"], dropna=False)
            .agg(row_count=("framework", "size"), error_count=("has_error", "sum"))
            .reset_index()
        )
    else:
        actual = pd.DataFrame(columns=["framework", "experiment", "native_modality", "row_count", "error_count"])

    coverage = expected.merge(actual, on=["framework", "experiment", "native_modality"], how="left")
    coverage["row_count"] = coverage["row_count"].fillna(0).astype(int)
    coverage["error_count"] = coverage["error_count"].fillna(0).astype(int)
    coverage["status"] = np.where(coverage["row_count"] == 0, "missing", np.where(coverage["error_count"] > 0, "error", "ok"))

    path = OUT_DIR / "experiment_coverage_report.csv"
    coverage.to_csv(path, index=False)
    print("Saved coverage report:", path)
    print("Expected rows:", len(expected))
    print("Actual rows:", len(final_results_df))
    print("Rows with errors:", int(coverage["error_count"].sum()))
    return coverage




def main():
    set_seed(SEED)

    print("=" * 80)
    print("Native tsai + aeon evaluation")
    print("=" * 80)
    print("CURRENT_MODEL_FAMILY:", CURRENT_MODEL_FAMILY)
    print("CURRENT_SYNTHETIC_METHOD:", CURRENT_SYNTHETIC_METHOD)
    print("CURRENT_SYNTHETIC_METHOD_DISPLAY:", CURRENT_SYNTHETIC_METHOD_DISPLAY)
    print("RUN_REAL_TO_REAL_ONLY:", RUN_REAL_TO_REAL_ONLY)
    print("USE_REAL_MODELS:", USE_REAL_MODELS)
    print("FRAMEWORKS:", FRAMEWORKS)
    print("MODALITIES:", MODALITIES)
    print("OUT_DIR:", OUT_DIR)
    print("REAL_DIR:", REAL_DIR)
    print("SYN_DIR:", SYN_DIR)
    print("PRETRAINED_RESULTS_DIR:", PRETRAINED_RESULTS_DIR)

    load_synthetic = not RUN_REAL_TO_REAL_ONLY
    real_X, real_y, real_subjects, syn_X, syn_y, syn_subjects = load_native_data(load_synthetic=load_synthetic)

    views = make_native_dataset_views(real_X, real_y, real_subjects, syn_X, syn_y, syn_subjects)
    shape_report_df = save_shape_report(views)
    safe_display(shape_report_df)

    all_results = []
    all_activity_results = []

    if "tsai" in FRAMEWORKS:
        tsai_results, tsai_activity = run_tsai(views, MODALITIES)
        all_results.append(tsai_results)
        all_activity_results.append(tsai_activity)

    if "aeon" in FRAMEWORKS:
        aeon_results, aeon_activity = run_aeon(views, MODALITIES)
        all_results.append(aeon_results)
        all_activity_results.append(aeon_activity)

    final_results = pd.concat(all_results, ignore_index=True) if all_results else pd.DataFrame()
    final_activity = pd.concat(all_activity_results, ignore_index=True) if all_activity_results else pd.DataFrame()

    final_results = add_method_columns(final_results)
    final_activity = add_method_columns(final_activity)

    final_results_path = OUT_DIR / "all_framework_native_results.csv"
    final_activity_path = OUT_DIR / "all_framework_native_per_activity_results.csv"
    final_results.to_csv(final_results_path, index=False)
    final_activity.to_csv(final_activity_path, index=False)
    coverage = save_coverage_report(final_results)

    summary = {
        "model_family": CURRENT_MODEL_FAMILY,
        "synthetic_method": CURRENT_SYNTHETIC_METHOD,
        "synthetic_method_display_name": CURRENT_SYNTHETIC_METHOD_DISPLAY,
        "frameworks": FRAMEWORKS,
        "modalities": MODALITIES,
        "run_real_to_real_only": RUN_REAL_TO_REAL_ONLY,
        "use_real_models": USE_REAL_MODELS,
        "epochs_tsai": EPOCHS_TSAI,
        "batch_size": BATCH_SIZE,
        "lr_tsai": LR_TSAI,
        "tsai_arch": TSAI_ARCH,
        "aeon_n_kernels": AEON_N_KERNELS,
        "aeon_n_jobs": AEON_N_JOBS,
        "aeon_fix_low_variation": AEON_FIX_LOW_VARIATION,
        "aeon_min_std": AEON_MIN_STD,
        "out_dir": str(OUT_DIR),
        "real_dir": str(REAL_DIR),
        "syn_dir": str(SYN_DIR),
        "pretrained_results_dir": str(PRETRAINED_RESULTS_DIR),
        "final_results_path": str(final_results_path),
        "final_activity_path": str(final_activity_path),
    }
    summary_path = OUT_DIR / "evaluation_summary.json"
    summary_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")

    print("\n" + "=" * 80)
    print("DONE")
    print("=" * 80)
    print("Saved combined aggregate results:", final_results_path)
    print("Saved combined per-activity results:", final_activity_path)
    print("Saved summary:", summary_path)

    print("\nAggregate results:")
    safe_display(final_results)
    print("\nCoverage:")
    safe_display(coverage)

    return {
        "final_results": final_results,
        "final_activity": final_activity,
        "coverage": coverage,
        "shape_report": shape_report_df,
        "summary_path": summary_path,
        "out_dir": OUT_DIR,
    }


def save_project_combined_outputs(method_outputs, project_root, output_root, model_family):
    output_root = Path(output_root)
    family_dir = output_root / model_family
    family_dir.mkdir(parents=True, exist_ok=True)

    result_frames = []
    activity_frames = []
    coverage_frames = []

    for method_name, outputs in method_outputs.items():
        if outputs is None:
            continue

        res = outputs.get("final_results", pd.DataFrame())
        act = outputs.get("final_activity", pd.DataFrame())
        cov = outputs.get("coverage", pd.DataFrame())

        if len(res) > 0:
            result_frames.append(res.copy())
        if len(act) > 0:
            activity_frames.append(act.copy())
        if len(cov) > 0:
            cov = cov.copy()
            cov.insert(0, "synthetic_method", method_name)
            cov.insert(0, "model_family", model_family)
            coverage_frames.append(cov)

    combined_results = pd.concat(result_frames, ignore_index=True) if result_frames else pd.DataFrame()
    combined_activity = pd.concat(activity_frames, ignore_index=True) if activity_frames else pd.DataFrame()
    combined_coverage = pd.concat(coverage_frames, ignore_index=True) if coverage_frames else pd.DataFrame()

    combined_results_path = family_dir / "combined_all_framework_native_results.csv"
    combined_activity_path = family_dir / "combined_all_framework_native_per_activity_results.csv"
    combined_coverage_path = family_dir / "combined_experiment_coverage_report.csv"
    compact_path = family_dir / "combined_compact_downstream_comparison.csv"
    ranking_path = family_dir / "combined_macro_f1_ranking.csv"

    combined_results.to_csv(combined_results_path, index=False)
    combined_activity.to_csv(combined_activity_path, index=False)
    combined_coverage.to_csv(combined_coverage_path, index=False)

    if len(combined_results) > 0:
        compact_cols = [
            "model_family",
            "synthetic_method",
            "synthetic_method_display_name",
            "framework",
            "model",
            "native_modality",
            "experiment",
            "accuracy",
            "macro_f1",
            "weighted_f1",
            "balanced_accuracy",
            "used_saved_real_to_real_model",
            "training_reused_from_experiment",
            "error",
        ]
        compact_cols = [c for c in compact_cols if c in combined_results.columns]
        compact = combined_results[compact_cols].copy()
        compact.to_csv(compact_path, index=False)

        if "macro_f1" in combined_results.columns:
            ranking = combined_results.dropna(subset=["macro_f1"]).sort_values(
                by=["native_modality", "macro_f1"],
                ascending=[True, False],
            )
            ranking.to_csv(ranking_path, index=False)
        else:
            pd.DataFrame().to_csv(ranking_path, index=False)
    else:
        pd.DataFrame().to_csv(compact_path, index=False)
        pd.DataFrame().to_csv(ranking_path, index=False)

    summary = {
        "model_family": model_family,
        "methods": list(method_outputs.keys()),
        "combined_results_path": str(combined_results_path),
        "combined_activity_path": str(combined_activity_path),
        "combined_coverage_path": str(combined_coverage_path),
        "compact_path": str(compact_path),
        "ranking_path": str(ranking_path),
    }
    summary_path = family_dir / "combined_evaluation_summary.json"
    summary_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")

    print("\nSaved family-level combined outputs:")
    print("  ", combined_results_path)
    print("  ", combined_activity_path)
    print("  ", combined_coverage_path)
    print("  ", compact_path)
    print("  ", ranking_path)
    print("  ", summary_path)

    return {
        "combined_results": combined_results,
        "combined_activity": combined_activity,
        "combined_coverage": combined_coverage,
        "combined_results_path": combined_results_path,
        "combined_activity_path": combined_activity_path,
        "combined_coverage_path": combined_coverage_path,
        "compact_path": compact_path,
        "ranking_path": ranking_path,
        "summary_path": summary_path,
    }


def run_project_evaluation(
    model_families=("kovae",),
    project_root="/home/iailab42/khans1/projects/ir",
    output_root=None,
    pretrained_results_dir=None,
):
    project_root = Path(project_root)

    if output_root is None:
        output_root = project_root / "results/downstream_extended_tsai_aeon"
    else:
        output_root = Path(output_root)

    if pretrained_results_dir is None:
        pretrained_results_dir = project_root / "models/downstream/pretrained/tsai_aeon_results"
    else:
        pretrained_results_dir = Path(pretrained_results_dir)

    all_family_outputs = {}

    for model_family in model_families:
        if model_family not in MODEL_FAMILY_CONFIGS:
            raise ValueError(f"Unknown model family: {model_family}")

        method_outputs = {}
        synthetic_methods = MODEL_FAMILY_CONFIGS[model_family]["synthetic_methods"]

        for synthetic_method in synthetic_methods:
            print("\n" + "#" * 100)
            print(f"RUNNING {model_family} | {synthetic_method}")
            print("#" * 100)

            configure_project_paths(
                project_root=project_root,
                model_family=model_family,
                synthetic_method=synthetic_method,
                output_root=output_root,
                pretrained_results_dir=pretrained_results_dir,
            )

            method_outputs[synthetic_method] = main()

        family_outputs = save_project_combined_outputs(
            method_outputs=method_outputs,
            project_root=project_root,
            output_root=output_root,
            model_family=model_family,
        )
        all_family_outputs[model_family] = {
            "method_outputs": method_outputs,
            "family_outputs": family_outputs,
        }

    return all_family_outputs



## Run cell

Set the frameworks, modalities, and model families here.

For the full friend-style evaluation use:

```python
FRAMEWORKS = ["tsai", "aeon"]
MODALITIES = ["acc", "bvp", "eda", "temp", "fused"]
```

For a faster test, use fewer modalities or only one framework.


In [ ]:
# =========================
# FINAL RUN CONFIG
# =========================

PROJECT_ROOT = Path("/home/iailab42/khans1/projects/ir")
PRETRAINED_RESULTS_DIR = PROJECT_ROOT / "models/downstream/pretrained/tsai_aeon_results"

# Friend-style full evaluation.
# FRAMEWORKS = ["tsai", "aeon"]
FRAMEWORKS = ["tsai"]
MODALITIES = ["acc", "bvp", "eda", "temp", "fused"]

# Keep pretrained model reuse.
USE_REAL_MODELS = True
RUN_REAL_TO_REAL_ONLY = False

# tsai settings from friend's notebook.
EPOCHS_TSAI = 50
BATCH_SIZE = 1024
LR_TSAI = 1e-3
TSAI_ARCH = "InceptionTimePlus"

# aeon settings from friend's notebook.
AEON_N_KERNELS = 5000
AEON_N_JOBS = 8

# Keep aeon compatibility fix.
AEON_FIX_LOW_VARIATION = True
AEON_MIN_STD = 1e-6

# Run both families, or choose only one:
# MODEL_FAMILIES_TO_RUN = ["kovae"]
# MODEL_FAMILIES_TO_RUN = ["timevae"]
MODEL_FAMILIES_TO_RUN = ["kovae", "timevae"]

print("Starting evaluation from final cell only.")
print("MODEL_FAMILIES_TO_RUN:", MODEL_FAMILIES_TO_RUN)
print("FRAMEWORKS:", FRAMEWORKS)
print("MODALITIES:", MODALITIES)

outputs = run_project_evaluation(
    model_families=MODEL_FAMILIES_TO_RUN,
    project_root=PROJECT_ROOT,
    output_root=PROJECT_ROOT / "results/downstream_extended_tsai_aeon",
    pretrained_results_dir=PRETRAINED_RESULTS_DIR,
)


Starting evaluation from final cell only.
MODEL_FAMILIES_TO_RUN: ['kovae', 'timevae']
FRAMEWORKS: ['tsai']
MODALITIES: ['acc', 'bvp', 'eda', 'temp', 'fused']

####################################################################################################
RUNNING kovae | rollout_v1
####################################################################################################

Configured project paths
  PROJECT_ROOT: /home/iailab42/khans1/projects/ir
  model_family: kovae
  synthetic_method: rollout_v1
  synthetic_method_display: KoVAE-Rollout
  REAL_DIR: /home/iailab42/khans1/projects/ir/data/processed/native_rates
  SYN_DIR: /home/iailab42/khans1/projects/ir/data/synthetic_subjects/kovae/rollout_v1
  OUT_DIR: /home/iailab42/khans1/projects/ir/results/downstream_extended_tsai_aeon/kovae/rollout_v1
  PRETRAINED_RESULTS_DIR: /home/iailab42/khans1/projects/ir/models/downstream/pretrained/tsai_aeon_results
Native tsai + aeon evaluation
CURRENT_MODEL_FAMILY: kovae
CURRENT_SYNTHETI

,native_view,native_hz,channels,native_real_train_shape_N_T_C,native_real_val_shape_N_T_C,native_real_test_shape_N_T_C,framework_real_train_shape_N_C_T,framework_real_val_shape_N_C_T,framework_real_test_shape_N_C_T,native_syn_all_shape_N_T_C,native_syn_test3_shape_N_T_C,syn_test_subjects,framework_syn_all_shape_N_C_T,framework_syn_test3_shape_N_C_T
0,acc,32,"ACC_x,ACC_y,ACC_z","[30762, 256, 3]","[6100, 256, 3]","[10063, 256, 3]","[30762, 3, 256]","[6100, 3, 256]","[10063, 3, 256]","[30000, 256, 3]","[9000, 256, 3]","synthetic_subject_01,synthetic_subject_02,synt...","[30000, 3, 256]","[9000, 3, 256]"
1,bvp,64,BVP,"[30762, 512, 1]","[6100, 512, 1]","[10063, 512, 1]","[30762, 1, 512]","[6100, 1, 512]","[10063, 1, 512]","[30000, 512, 1]","[9000, 512, 1]","synthetic_subject_01,synthetic_subject_02,synt...","[30000, 1, 512]","[9000, 1, 512]"
2,eda,4,EDA,"[30762, 32, 1]","[6100, 32, 1]","[10063, 32, 1]","[30762, 1, 32]","[6100, 1, 32]","[10063, 1, 32]","[30000, 32, 1]","[9000, 32, 1]","synthetic_subject_01,synthetic_subject_02,synt...","[30000, 1, 32]","[9000, 1, 32]"
3,temp,4,TEMP,"[30762, 32, 1]","[6100, 32, 1]","[10063, 32, 1]","[30762, 1, 32]","[6100, 1, 32]","[10063, 1, 32]","[30000, 32, 1]","[9000, 32, 1]","synthetic_subject_01,synthetic_subject_02,synt...","[30000, 1, 32]","[9000, 1, 32]"
4,fused,64,"ACC_x,ACC_y,ACC_z,BVP,EDA,TEMP","[30762, 512, 6]","[6100, 512, 6]","[10063, 512, 6]","[30762, 6, 512]","[6100, 6, 512]","[10063, 6, 512]","[30000, 512, 6]","[9000, 512, 6]","synthetic_subject_01,synthetic_subject_02,synt...","[30000, 6, 512]","[9000, 6, 512]"



[tsai] real_to_real | acc
Reusing saved tsai Real->Real metrics.

[tsai] real_to_synthetic | acc
Copied external tsai model into current output folder: /home/iailab42/khans1/projects/ir/results/downstream_extended_tsai_aeon/kovae/rollout_v1/tsai_saved_models/best_InceptionTimePlus_real_to_real_acc.pth
Loading tsai Real->Real model: /home/iailab42/khans1/projects/ir/results/downstream_extended_tsai_aeon/kovae/rollout_v1/tsai_saved_models/best_InceptionTimePlus_real_to_real_acc.pth


Saved: /home/iailab42/khans1/projects/ir/results/downstream_extended_tsai_aeon/kovae/rollout_v1/tsai_real_to_synthetic_acc_predictions.csv
Saved: /home/iailab42/khans1/projects/ir/results/downstream_extended_tsai_aeon/kovae/rollout_v1/tsai_real_to_synthetic_acc_confusion_matrix.csv
Saved: /home/iailab42/khans1/projects/ir/results/downstream_extended_tsai_aeon/kovae/rollout_v1/tsai_real_to_synthetic_acc_confusion_matrix_counts.png
Saved: /home/iailab42/khans1/projects/ir/results/downstream_extended_tsai_aeon/kovae/rollout_v1/tsai_real_to_synthetic_acc_confusion_matrix_normalized.png
Saved: /home/iailab42/khans1/projects/ir/results/downstream_extended_tsai_aeon/kovae/rollout_v1/tsai_real_to_synthetic_acc_per_activity_metrics.csv
{
  "framework": "tsai",
  "model": "InceptionTimePlus",
  "experiment": "real_to_synthetic",
  "native_modality": "acc",
  "native_hz": 32,
  "channels": "ACC_x,ACC_y,ACC_z",
  "description": "ACC only",
  "input_shape_train_native_N_T_C": [
    30762,
    256,


epoch,train_loss,valid_loss,accuracy,time
0,1.490812,2.780220,0.365410,00:07
1,1.104810,8.537931,0.305410,00:07
2,0.800309,12.908846,0.319180,00:07
3,0.544108,17.179090,0.266885,00:07
4,0.361470,20.539274,0.232951,00:07
5,0.242827,29.138273,0.231967,00:07
6,0.169234,17.310942,0.251475,00:07
7,0.119997,23.323381,0.218852,00:07
8,0.086896,36.230591,0.237377,00:07
9,0.065899,29.229647,0.239672,00:07


...

Better model found at epoch 0 with accuracy value: 0.3581967353820801.
Better model found at epoch 1 with accuracy value: 0.4078688621520996.


Saved: /home/iailab42/khans1/projects/ir/results/downstream_extended_tsai_aeon/kovae/posterior_bank_v2/tsai_real_plus_synthetic_to_real_temp_predictions.csv
Saved: /home/iailab42/khans1/projects/ir/results/downstream_extended_tsai_aeon/kovae/posterior_bank_v2/tsai_real_plus_synthetic_to_real_temp_confusion_matrix.csv
Saved: /home/iailab42/khans1/projects/ir/results/downstream_extended_tsai_aeon/kovae/posterior_bank_v2/tsai_real_plus_synthetic_to_real_temp_confusion_matrix_counts.png
Saved: /home/iailab42/khans1/projects/ir/results/downstream_extended_tsai_aeon/kovae/posterior_bank_v2/tsai_real_plus_synthetic_to_real_temp_confusion_matrix_normalized.png
Saved: /home/iailab42/khans1/projects/ir/results/downstream_extended_tsai_aeon/kovae/posterior_bank_v2/tsai_real_plus_synthetic_to_real_temp_per_activity_metrics.csv
{
  "framework": "tsai",
  "model": "InceptionTimePlus",
  "experiment": "real_plus_synthetic_to_real",
  "native_modality": "temp",
  "native_hz": 4,
  "channels": "TEMP",


Saved: /home/iailab42/khans1/projects/ir/results/downstream_extended_tsai_aeon/kovae/posterior_bank_v2/tsai_real_to_synthetic_fused_predictions.csv
Saved: /home/iailab42/khans1/projects/ir/results/downstream_extended_tsai_aeon/kovae/posterior_bank_v2/tsai_real_to_synthetic_fused_confusion_matrix.csv
Saved: /home/iailab42/khans1/projects/ir/results/downstream_extended_tsai_aeon/kovae/posterior_bank_v2/tsai_real_to_synthetic_fused_confusion_matrix_counts.png
Saved: /home/iailab42/khans1/projects/ir/results/downstream_extended_tsai_aeon/kovae/posterior_bank_v2/tsai_real_to_synthetic_fused_confusion_matrix_normalized.png
Saved: /home/iailab42/khans1/projects/ir/results/downstream_extended_tsai_aeon/kovae/posterior_bank_v2/tsai_real_to_synthetic_fused_per_activity_metrics.csv
{
  "framework": "tsai",
  "model": "InceptionTimePlus",
  "experiment": "real_to_synthetic",
  "native_modality": "fused",
  "native_hz": 64,
  "channels": "ACC_x,ACC_y,ACC_z,BVP,EDA,TEMP",
  "description": "ACC+BVP+E

epoch,train_loss,valid_loss,accuracy,time
0,1.469365,1.499293,0.681639,00:06
1,1.097378,0.894736,0.744590,00:06
2,0.856090,0.762600,0.744262,00:06
3,0.687649,0.656238,0.760000,00:06
4,0.567964,0.728294,0.732131,00:06
5,0.484262,0.704745,0.773443,00:06
6,0.424306,0.680822,0.747213,00:06
7,0.378036,0.725990,0.755410,00:06
8,0.346398,0.695414,0.755738,00:06
9,0.323037,0.630804,0.749672,00:06


Better model found at epoch 0 with accuracy value: 0.6816393733024597.
Better model found at epoch 1 with accuracy value: 0.744590163230896.
Better model found at epoch 3 with accuracy value: 0.7599999904632568.
Better model found at epoch 5 with accuracy value: 0.7734426259994507.
Better model found at epoch 27 with accuracy value: 0.7924590110778809.
